# 1. Load data

In [47]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [48]:
df = pd.read_csv("carclaims.csv")

In [49]:
df.shape

(15420, 33)

In [50]:
df.head()

,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,...,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange-Claim,NumberOfCars,Year,BasePolicy,FraudFound
0,Dec,5,Wednesday,Honda,Urban,Tuesday,Jan,1,Female,Single,...,26 to 30,No,No,External,none,1 year,3 to 4,1994,Liability,No
1,Jan,3,Wednesday,Honda,Urban,Monday,Jan,4,Male,Single,...,31 to 35,Yes,No,External,none,no change,1 vehicle,1994,Collision,No
2,Oct,5,Friday,Honda,Urban,Thursday,Nov,2,Male,Married,...,41 to 50,No,No,External,none,no change,1 vehicle,1994,Collision,No
3,Jun,2,Saturday,Toyota,Rural,Friday,Jul,1,Male,Married,...,51 to 65,Yes,No,External,more than 5,no change,1 vehicle,1994,Liability,No
4,Jan,5,Monday,Honda,Urban,Tuesday,Feb,2,Female,Single,...,31 to 35,No,No,External,none,no change,1 vehicle,1994,Collision,No


# 2. Clean columns names (cleaning)

In [51]:
df.columns = df.columns.str.lower().str.replace('[^a-z0-9]', '_', regex=True)
print(df.columns.tolist())

['month', 'weekofmonth', 'dayofweek', 'make', 'accidentarea', 'dayofweekclaimed', 'monthclaimed', 'weekofmonthclaimed', 'sex', 'maritalstatus', 'age', 'fault', 'policytype', 'vehiclecategory', 'vehicleprice', 'policynumber', 'repnumber', 'deductible', 'driverrating', 'days_policy_accident', 'days_policy_claim', 'pastnumberofclaims', 'ageofvehicle', 'ageofpolicyholder', 'policereportfiled', 'witnesspresent', 'agenttype', 'numberofsuppliments', 'addresschange_claim', 'numberofcars', 'year', 'basepolicy', 'fraudfound']


# 3. Drop Leakage and Useless Columns

In [53]:
drop_cols = [
    'policynumber',
    'repnumber',
    'daysofpolicyclaimed',
    'dayofweekclaimed',
    'monthclaimed',
    'weekofmonthclaimed',
    'days_policy_claim',
]

drop_cols = [c for c in drop_cols if c in df.columns]
df.drop(columns=drop_cols, inplace=True)
print("Remaining shape:", df.shape)

Remaining shape: (15420, 27)


In [54]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15420 entries, 0 to 15419
Data columns (total 27 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   month                 15420 non-null  object
 1   weekofmonth           15420 non-null  int64 
 2   dayofweek             15420 non-null  object
 3   make                  15420 non-null  object
 4   accidentarea          15420 non-null  object
 5   sex                   15420 non-null  object
 6   maritalstatus         15420 non-null  object
 7   age                   15420 non-null  int64 
 8   fault                 15420 non-null  object
 9   policytype            15420 non-null  object
 10  vehiclecategory       15420 non-null  object
 11  vehicleprice          15420 non-null  object
 12  deductible            15420 non-null  int64 
 13  driverrating          15420 non-null  int64 
 14  days_policy_accident  15420 non-null  object
 15  pastnumberofclaims    15420 non-null

# Handle Missing Values & Fix Errors

In [55]:
df[df["age"] == 0]["fraudfound"].value_counts()

fraudfound
No     289
Yes     31
Name: count, dtype: int64

In [56]:
(df["age"] == 0).sum()

np.int64(320)

In [57]:
df['is_age_placeholder'] = (df['age'] == 0).astype(int)
df['age'] = df['age'].replace(0, np.nan)

# Fill missing age based on the median age of people with the same Marital Status and Vehicle Price
df['age'] = df['age'].fillna(df.groupby(['maritalstatus', 'vehicleprice'])['age'].transform('median'))
# If any are still empty, use the global median
df['age'] = df['age'].fillna(df['age'].median())

(df["age"] == 0).sum()


np.int64(0)

# 5. Feature engineering (new columns)

### High claim history

In [ ]:
df["high_claim_history"] = df["pastnumberofclaims"].isin(
    ["2 to 4", "more than 4"]
).astype(int)

### Weekend accident


In [ ]:
df["weekend_accident"] = df["dayofweek"].isin(
    ["Saturday", "Sunday"]
).astype(int)

### Old vehicle


In [ ]:
df["old_vehicle"] = df["ageofvehicle"].isin(
    ["7 years", "more than 7"]
).astype(int)

### High deductible


In [ ]:
df["high_deductible"] = (df["deductible"] > 400).astype(int)

### Multiple cars


In [ ]:
df["multiple_cars"] = (df["numberofcars"] != "1 vehicle").astype(int)

# 6. Encode categoricals